# IAN 603 · Module 6
# Joining Data in PostgreSQL

A join does two things:

1. decides **which rows match**, and
2. decides **which unmatched rows should still be kept**.

The most important habit in this module is:

> **Define the target grain before you join.**

Use this workflow:

```text
Define one output row
        ↓
Identify table relationships
        ↓
Choose which population must be preserved
        ↓
Write the match condition
        ↓
Inspect row multiplication
        ↓
Validate counts and unmatched rows
```

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module6.html


## What you will learn

By the end of this notebook, you should be able to:

1. Explain the difference between matching rows and preserving rows.
2. Use `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, and `FULL OUTER JOIN`.
3. Find unmatched rows with an anti-join pattern.
4. Explain why filtering in `ON` can differ from filtering in `WHERE`.
5. Use `USING` when the join key has the same name on both sides.
6. Use self joins for same-table relationships.
7. Explain when a `CROSS JOIN` is appropriate.
8. Identify one-to-one, one-to-many, and many-to-many row growth.
9. Recognize the `COUNT(*)` trap after joining several child tables.
10. Use `COUNT(DISTINCT ...)` when appropriate.
11. Pre-aggregate child tables before joining when the target grain requires it.
12. Validate row counts and unmatched keys after a join.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Meet the Relationship Model

This module uses related clinical-style tables.

```text
patients
   │
   ├────< admissions >──── medical_staff
   │             │
   │             └──────── departments
   │
   ├────< lab_results
   │
   ├────< prescriptions
   │
   └────< patient_vitals

departments ───< medical_staff

admissions_stage
   └── may contain patient IDs that do not exist in patients
```

### Source grain

| Table | One row represents |
|---|---|
| `patients` | one patient |
| `admissions` | one admission |
| `lab_results` | one lab test |
| `prescriptions` | one prescription |
| `medical_staff` | one staff member |
| `departments` | one department |
| `admissions_stage` | one imported staging row |


## 4. Create or reset the Module 6 teaching database


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 6\n-- Joining Data in PostgreSQL\n-- ============================================================\n-- Companion file for s_module6.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- INNER, LEFT, RIGHT, FULL OUTER JOIN, anti-joins, ON vs WHERE,\n-- USING, self joins, cross joins, cardinality, pre-aggregation,\n-- and join validation.\n--\n-- Main habit: define what one output row should represent before\n-- choosing and writing the join.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS appointment_slots CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS patient_vitals CASCADE;\nDROP TABLE IF EXISTS prescriptions CASCADE;\nDROP TABLE IF EXISTS lab_results CASCADE;\nDROP TABLE IF EXISTS admissions_stage CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS medical_staff CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    patient_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    gender varchar(20),\n    home_city varchar(60) NOT NULL\n);\n\nCREATE TABLE departments (\n    department varchar(60) PRIMARY KEY,\n    floor_no integer NOT NULL,\n    service_line varchar(80) NOT NULL\n);\n\nCREATE TABLE medical_staff (\n    staff_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    department varchar(60) REFERENCES departments(department),\n    specialization varchar(80) NOT NULL,\n    hire_date date NOT NULL\n);\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    admit_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(120) NOT NULL,\n    department varchar(60) NOT NULL REFERENCES departments(department),\n    attending_staff_id integer REFERENCES medical_staff(staff_id),\n    CHECK (discharge_date IS NULL OR discharge_date >= admit_date)\n);\n\n-- Staging data intentionally has no foreign key so students can see an orphan row.\nCREATE TABLE admissions_stage (\n    import_row_id integer PRIMARY KEY,\n    patient_id integer,\n    diagnosis varchar(120),\n    source_file varchar(80)\n);\n\nCREATE TABLE lab_results (\n    test_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    test_name varchar(60) NOT NULL,\n    result_value numeric(10,2),\n    test_date date NOT NULL,\n    normal_range varchar(30)\n);\n\nCREATE TABLE prescriptions (\n    rx_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    medication varchar(100) NOT NULL,\n    dosage varchar(50) NOT NULL,\n    prescribed_date date NOT NULL,\n    duration_days integer CHECK (duration_days > 0)\n);\n\nCREATE TABLE patient_vitals (\n    vital_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    vital_name varchar(50) NOT NULL,\n    vital_value numeric(10,2),\n    recorded_date date NOT NULL\n);\n\nCREATE TABLE appointment_slots (\n    slot_label varchar(30) PRIMARY KEY\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith',45,'Female','Greensboro'),\n(1002,'Bob Johnson',32,'Male','High Point'),\n(1003,'Carol White',58,'Female','Greensboro'),\n(1004,'David Brown',67,'Male','Burlington'),\n(1005,'Eve Davis',29,'Female','Greensboro'),\n(1006,'Frank Miller',71,'Male','Reidsville'),\n(1007,'Grace Wilson',52,'Female','Greensboro'),\n(1008,'Henry Taylor',48,'Male','High Point'),\n(1009,'Ivy Clark',63,'Female','Greensboro'),\n(1010,'Jack Anderson',55,'Male','Burlington'),\n(1011,'Kara Young',41,'Female','Greensboro');\n\nINSERT INTO departments VALUES\n('Cardiology',3,'Cardiovascular'),\n('Endocrinology',4,'Medicine'),\n('Internal Medicine',2,'Medicine'),\n('Neurology',5,'Neurosciences'),\n('Orthopedics',1,'Musculoskeletal'),\n('Pulmonology',4,'Medicine'),\n('Dermatology',6,'Specialty Care');\n\nINSERT INTO medical_staff VALUES\n(501,'Dr. Sarah Chen','Cardiology','Cardiologist','2020-03-15'),\n(502,'Dr. Michael Rodriguez','Internal Medicine','Internist','2019-07-22'),\n(503,'Dr. Emily Watson','Orthopedics','Orthopedic Surgeon','2021-01-10'),\n(504,'Dr. James Kim','Neurology','Neurologist','2018-11-30'),\n(505,'Dr. Lisa Thompson','Endocrinology','Endocrinologist','2022-05-18'),\n(506,'Dr. Robert Brown','Pulmonology','Pulmonologist','2020-09-05'),\n(507,'Dr. Amanda Garcia','Cardiology','Cardiac Electrophysiologist','2021-08-12');\n\nINSERT INTO admissions VALUES\n(101,1001,'2026-01-15','2026-01-20','Pneumonia','Internal Medicine',502),\n(102,1002,'2026-01-16','2026-01-18','Fractured arm','Orthopedics',503),\n(103,1003,'2026-01-17','2026-01-25','Cardiac arrhythmia','Cardiology',501),\n(104,1001,'2026-03-05','2026-03-12','Hypertension','Internal Medicine',502),\n(105,1004,'2026-02-19','2026-02-26','Heart disease','Cardiology',507),\n(106,1005,'2026-04-20','2026-04-22','Migraine','Neurology',504),\n(107,1006,'2026-05-21','2026-05-28','Diabetes complications','Endocrinology',505),\n(108,1007,'2026-05-22','2026-05-24','Asthma','Pulmonology',506),\n(109,1008,'2026-06-23','2026-06-30','Cardiac monitoring','Cardiology',501),\n(110,1009,'2026-07-24','2026-07-27','Joint pain','Orthopedics',503),\n(111,1010,'2026-07-25',NULL,'Hypertension','Internal Medicine',502),\n(112,1006,'2026-08-01','2026-08-05','Cardiac assessment','Cardiology',507);\n\nINSERT INTO admissions_stage VALUES\n(9001,1001,'Pneumonia','july_import.csv'),\n(9002,1010,'Hypertension','july_import.csv'),\n(9003,1999,'External transfer','july_import.csv');\n\nINSERT INTO lab_results VALUES\n(201,1001,'Glucose',98,'2026-01-16','70-110'),\n(202,1002,'Cholesterol',185,'2026-01-17','<200'),\n(203,1003,'Glucose',145,'2026-01-18','70-110'),\n(204,1004,'Blood Pressure',150,'2026-02-19','<140'),\n(205,1005,'Cholesterol',210,'2026-04-20','<200'),\n(206,1006,'Glucose',320,'2026-05-21','70-110'),\n(207,1007,'Blood Pressure',128,'2026-05-22','<140'),\n(208,1008,'Cholesterol',195,'2026-06-23','<200'),\n(209,1009,'Glucose',110,'2026-07-24','70-110'),\n(210,1010,'Blood Pressure',142,'2026-07-25','<140'),\n(211,1001,'Hemoglobin',14.2,'2026-01-16','12-16'),\n(212,1003,'Hemoglobin',11.8,'2026-01-18','12-16');\n\nINSERT INTO prescriptions VALUES\n(301,1001,'Amoxicillin','500mg','2026-01-16',7),\n(302,1002,'Ibuprofen','400mg','2026-01-17',14),\n(303,1003,'Metoprolol','25mg','2026-01-18',90),\n(304,1004,'Atorvastatin','40mg','2026-02-19',30),\n(305,1006,'Metformin','1000mg','2026-05-21',60),\n(306,1006,'Insulin','50 units','2026-05-21',90),\n(307,1001,'Lisinopril','5mg','2026-03-06',30),\n(308,1007,'Albuterol','2 puffs','2026-05-22',15),\n(309,1008,'Aspirin','81mg','2026-06-23',365),\n(310,1009,'Ibuprofen','400mg','2026-07-24',7);\n\nINSERT INTO patient_vitals VALUES\n(401,1001,'Heart Rate',88,'2026-01-15'),\n(402,1001,'Temperature C',38.4,'2026-01-15'),\n(403,1003,'Heart Rate',104,'2026-01-17'),\n(404,1004,'Systolic BP',150,'2026-02-19'),\n(405,1006,'Glucose',320,'2026-05-21');\n\nINSERT INTO appointment_slots VALUES\n('08:00'),('10:00'),('13:00');"

execute_script(setup_sql)


## 5. Preview patients


In [ ]:
display_query("""
SELECT patient_id, name, age, gender, home_city
FROM patients
ORDER BY patient_id;
""", title="Patients")


## 6. Preview admissions


In [ ]:
display_query("""
SELECT
    admission_id,
    patient_id,
    admit_date,
    discharge_date,
    diagnosis,
    department,
    attending_staff_id
FROM admissions
ORDER BY admission_id;
""", title="Admissions")


# Part II · The Join Mental Model

Suppose we have two tiny tables.

### Patients

| patient_id | name |
|---:|---|
| 1 | A |
| 2 | B |
| 3 | C |

### Admissions

| admission_id | patient_id |
|---:|---:|
| 101 | 1 |
| 102 | 1 |
| 103 | 3 |

The join condition:

```sql
ON a.patient_id = p.patient_id
```

answers:

> Which rows belong together?

The join type answers:

> What should happen to rows that do not find a match?


## Join preservation map

| Join | Matched rows | Unmatched left | Unmatched right |
|---|:---:|:---:|:---:|
| INNER | ✅ | ❌ | ❌ |
| LEFT | ✅ | ✅ | ❌ |
| RIGHT | ✅ | ❌ | ✅ |
| FULL OUTER | ✅ | ✅ | ✅ |

Before writing the query, decide which population must be preserved.


# Part III · INNER JOIN

Use an `INNER JOIN` when the question requires rows that exist on both sides.

Question:

> Show admissions with their patient names.

Target grain:

> **one row per admission**

Because every valid admission belongs to one patient, the join should preserve admission grain.


## 7. Patients matched to admissions


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    a.admission_id,
    a.diagnosis,
    a.department
FROM patients AS p
INNER JOIN admissions AS a
    ON a.patient_id = p.patient_id
ORDER BY p.patient_id, a.admit_date;
""", title="INNER JOIN: matched patient-admission rows")


### What happened to the grain?

`patients` has one row per patient.

`admissions` has one row per admission.

Because a patient can have several admissions, the joined result is:

> **one row per admission**

Patient columns repeat when the patient has multiple admissions.

That repetition is expected.


# Part IV · LEFT JOIN

Use a `LEFT JOIN` when all rows from the left table must remain.

Question:

> Show every patient, along with any admissions they may have.

Target population:

> **all patients**

A patient with no admission should still appear, with NULL values from the admission side.


## 8. Preserve every patient


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    a.admission_id,
    a.diagnosis
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.patient_id = p.patient_id
ORDER BY p.patient_id, a.admission_id;
""", title="LEFT JOIN: preserve all patients")


### Read NULLs after a LEFT JOIN

If a patient has:

| patient_id | name | admission_id | diagnosis |
|---:|---|---:|---|
| 1008 | Henry Taylor | NULL | NULL |

that does **not** mean the patient row is missing.

It means:

> the patient was preserved, but no matching admission was found.


# Part V · Anti-Join: Find What Is Missing

A common analytical question is:

> Which left-side entities have **no** matching child row?

Pattern:

```sql
LEFT JOIN child ...
WHERE child.primary_key IS NULL
```

This is often called an **anti-join pattern**.


## 9. Find patients with no lab result


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name
FROM patients AS p
LEFT JOIN lab_results AS l
    ON l.patient_id = p.patient_id
WHERE l.test_id IS NULL
ORDER BY p.patient_id;
""", title="Patients with no lab result")


### Why test the child key?

Use a child column that should never be NULL for a real matched row, such as the child primary key.

That makes:

```sql
WHERE l.test_id IS NULL
```

a clear signal that no child row matched.


# Part VI · RIGHT JOIN and FULL OUTER JOIN

These join types are valid, but they are often less common in everyday analytics.

A `RIGHT JOIN` preserves the right table.

A `FULL OUTER JOIN` preserves unmatched rows from both sides.


## 10. RIGHT JOIN: preserve every department


In [ ]:
display_query("""
SELECT
    d.department,
    d.floor_no,
    s.name AS staff_name,
    s.specialization
FROM medical_staff AS s
RIGHT JOIN departments AS d
    ON d.department = s.department
ORDER BY d.department, s.name;
""", title="RIGHT JOIN: preserve departments")


### Equivalent style

Many analysts find this easier to read as:

```sql
FROM departments AS d
LEFT JOIN medical_staff AS s
    ON ...
```

If the same meaning can be expressed with a `LEFT JOIN`, choose whichever direction makes the preserved population easiest to see.


## 11. FULL OUTER JOIN: reconcile master and staging data


In [ ]:
display_query("""
SELECT
    COALESCE(p.patient_id, s.patient_id) AS patient_id,
    p.name,
    s.import_row_id,
    s.diagnosis,
    CASE
        WHEN p.patient_id IS NULL
            THEN 'Staging row has no patient master'
        WHEN s.import_row_id IS NULL
            THEN 'Patient has no staging row'
        ELSE 'Matched'
    END AS match_status
FROM patients AS p
FULL OUTER JOIN admissions_stage AS s
    ON s.patient_id = p.patient_id
ORDER BY COALESCE(p.patient_id, s.patient_id);
""", title="FULL OUTER JOIN reconciliation")


### Why FULL OUTER JOIN is useful here

For reconciliation, we want to see:

- matched IDs,
- patient-master rows with no staging row,
- staging rows with no patient master.

A full join makes all three visible in one result.


# Part VII · ON vs WHERE with an Outer Join

This is one of the most important join concepts.

Suppose the goal is:

> Keep **every patient**, but attach only Glucose lab results.

There are two places we might write the lab condition:

- inside `ON`
- inside `WHERE`

They do not mean the same thing after a `LEFT JOIN`.


## 12. Filter the child rows inside ON

The patient population is preserved first.

Only matching Glucose lab rows are attached.


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    l.test_id,
    l.result_value
FROM patients AS p
LEFT JOIN lab_results AS l
    ON l.patient_id = p.patient_id
   AND l.test_name = 'Glucose'
ORDER BY p.patient_id;
""", title="LEFT JOIN with child filter in ON")


### Conceptual result

| patient | glucose result |
|---|---:|
| A | 96 |
| B | NULL |
| C | 104 |
| D | NULL |

Patients without a Glucose test stay in the result.


## 13. Put the same filter in WHERE

Now the `WHERE` clause removes rows where the joined lab is not Glucose.

That also removes patients whose lab side is NULL.


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    l.test_id,
    l.result_value
FROM patients AS p
LEFT JOIN lab_results AS l
    ON l.patient_id = p.patient_id
WHERE l.test_name = 'Glucose'
ORDER BY p.patient_id;
""", title="LEFT JOIN with child filter in WHERE")


### The practical rule

If your goal is:

> Preserve all left-side rows, but limit which right-side rows may match,

put the right-side condition in the `ON` clause.

A right-table condition in `WHERE` can effectively turn a `LEFT JOIN` into an inner-style result.


# Part VIII · USING

When the join key has the same name in both tables, PostgreSQL can use:

```sql
JOIN admissions USING (patient_id)
```

instead of:

```sql
JOIN admissions
    ON admissions.patient_id = patients.patient_id
```

`USING` also returns one join-key column instead of two.


## 14. Join with USING


In [ ]:
display_query("""
SELECT
    patient_id,
    p.name,
    a.admission_id,
    a.diagnosis
FROM patients AS p
JOIN admissions AS a USING (patient_id)
ORDER BY patient_id, a.admission_id;
""", title="JOIN ... USING")


Use `USING` when it improves readability.

Use `ON` when:
- join-column names differ,
- multiple conditions are required,
- the matching logic needs to be explicit.


# Part IX · Self Join

A self join joins a table to itself using different aliases.

This is useful when rows in the same table relate to one another.


## 15. Staff members in the same department


In [ ]:
display_query("""
SELECT
    s1.department,
    s1.name AS staff_member,
    s2.name AS colleague
FROM medical_staff AS s1
JOIN medical_staff AS s2
    ON s1.department = s2.department
   AND s1.staff_id < s2.staff_id
ORDER BY s1.department, s1.name;
""", title="Self join: colleagues")


### Why `s1.staff_id < s2.staff_id`?

Without it, we would get:

- A paired with B,
- B paired with A,
- and possibly A paired with A.

The inequality keeps each pair once and prevents self-pairs.


# Part X · Self Join for Repeat Admissions

A self join can also compare earlier and later events for the same patient.

Target grain:

> **one row per qualifying admission pair**


## 16. Admissions occurring within 90 days of an earlier discharge


In [ ]:
display_query("""
SELECT
    a1.patient_id,
    p.name,
    a1.admission_id AS earlier_admission,
    a2.admission_id AS later_admission,
    a2.admit_date - a1.discharge_date AS days_between
FROM admissions AS a1
JOIN admissions AS a2
    ON a1.patient_id = a2.patient_id
   AND a1.admission_id < a2.admission_id
JOIN patients AS p
    ON p.patient_id = a1.patient_id
WHERE a1.discharge_date IS NOT NULL
  AND a2.admit_date - a1.discharge_date BETWEEN 1 AND 90
ORDER BY days_between;
""", title="Potential repeat-admission pairs")


This approach compares **all qualifying pairs**.

Later, window functions such as `LAG` and `LEAD` can be more convenient when you specifically want the immediately previous or next event.


# Part XI · CROSS JOIN

A `CROSS JOIN` creates every combination of rows.

If table A has 2 rows and table B has 3 rows:

```text
2 × 3 = 6 rows
```

That growth is intentional only when the question truly needs combinations.


## 17. Departments × appointment slots


In [ ]:
display_query("""
SELECT
    d.department,
    s.slot_label
FROM departments AS d
CROSS JOIN appointment_slots AS s
WHERE d.department IN ('Cardiology', 'Internal Medicine')
ORDER BY d.department, s.slot_label;
""", title="CROSS JOIN combinations")


### A good CROSS JOIN question

> Create every department-slot combination for scheduling.

### A bad CROSS JOIN accident

Forgetting the join condition between two unrelated tables.

Always estimate the expected row count before using a cross join.


# Part XII · Cardinality and Row Multiplication

Before joining two tables, ask about the relationship.

| Relationship | Possible result |
|---|---|
| one-to-one | usually one joined row per parent |
| one-to-many | parent row repeats for each child |
| many-to-many | rows can multiply rapidly |

The dangerous case in analytics is joining **several one-to-many child tables** directly at the parent level.


## Example: one patient has

- 2 admissions
- 3 labs
- 2 prescriptions

A direct join can produce:

```text
2 × 3 × 2 = 12 joined rows
```

That does not mean the patient suddenly has 12 admissions.

It means several child relationships were combined into one row set.


## 18. Observe row multiplication


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT a.admission_id) AS admissions,
    COUNT(DISTINCT l.test_id) AS labs,
    COUNT(DISTINCT pr.rx_id) AS prescriptions
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.patient_id = p.patient_id
LEFT JOIN lab_results AS l
    ON l.patient_id = p.patient_id
LEFT JOIN prescriptions AS pr
    ON pr.patient_id = p.patient_id
GROUP BY p.patient_id, p.name
ORDER BY joined_rows DESC, p.patient_id;
""", title="Direct multi-child join")


# Part XIII · The COUNT Trap

After row multiplication:

```sql
COUNT(a.admission_id)
```

may count the same admission several times.

Compare it with:

```sql
COUNT(DISTINCT a.admission_id)
```


## 19. Naive count vs distinct count


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    COUNT(*) AS joined_rows,
    COUNT(a.admission_id) AS naive_admission_count,
    COUNT(DISTINCT a.admission_id) AS true_admissions
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.patient_id = p.patient_id
LEFT JOIN lab_results AS l
    ON l.patient_id = p.patient_id
LEFT JOIN prescriptions AS pr
    ON pr.patient_id = p.patient_id
GROUP BY p.patient_id, p.name
ORDER BY joined_rows DESC, p.patient_id;
""", title="COUNT trap after row multiplication")


### Important nuance

`COUNT(DISTINCT ...)` can correct a count, but it does not automatically fix every measure.

For example, summing admission charges after a row-multiplying join could still duplicate money.

When the target grain is one row per patient, a stronger design is often:

> **aggregate each child table to patient grain first, then join the summaries.**


# Part XIV · Pre-Aggregate Before Joining

Target grain:

> **one row per patient**

So first convert every child table to one row per patient.


## 20. Step 1 — admission counts

| patient_id | admissions |
|---:|---:|
| 1001 | 2 |
| 1002 | 1 |
| ... | ... |


In [ ]:
display_query("""
SELECT
    patient_id,
    COUNT(*) AS admissions
FROM admissions
GROUP BY patient_id
ORDER BY patient_id;
""", title="Admission counts at patient grain")


## 21. Step 2 — join patient-level summaries


In [ ]:
display_query("""
WITH admission_counts AS (
    SELECT
        patient_id,
        COUNT(*) AS admissions
    FROM admissions
    GROUP BY patient_id
),
lab_counts AS (
    SELECT
        patient_id,
        COUNT(*) AS labs
    FROM lab_results
    GROUP BY patient_id
),
rx_counts AS (
    SELECT
        patient_id,
        COUNT(*) AS prescriptions
    FROM prescriptions
    GROUP BY patient_id
)
SELECT
    p.patient_id,
    p.name,
    COALESCE(a.admissions, 0) AS admissions,
    COALESCE(l.labs, 0) AS labs,
    COALESCE(r.prescriptions, 0) AS prescriptions
FROM patients AS p
LEFT JOIN admission_counts AS a USING (patient_id)
LEFT JOIN lab_counts AS l USING (patient_id)
LEFT JOIN rx_counts AS r USING (patient_id)
ORDER BY p.patient_id;
""", title="Safe patient-level feature table")


### Why this is safer

Each CTE has:

> **one row per patient**

So joining them to `patients` preserves one row per patient.

No child table can multiply another child table.


# Part XV · Multiple Joins with a Clear Target Grain

Not every multi-table join is dangerous.

The question is whether the relationships preserve your intended grain.

Suppose the target grain is:

> **one row per admission**

We can join:
- patient information,
- admission information,
- one attending clinician.

If each admission has at most one attending staff ID, the admission grain remains stable.


## 22. Admission-level analytical dataset


In [ ]:
display_query("""
SELECT
    p.name,
    a.admission_id,
    a.diagnosis,
    a.department,
    ms.name AS attending_clinician,
    ms.specialization
FROM patients AS p
JOIN admissions AS a
    ON a.patient_id = p.patient_id
LEFT JOIN medical_staff AS ms
    ON ms.staff_id = a.attending_staff_id
ORDER BY a.admit_date, p.name;
""", title="One row per admission")


# Part XVI · Department Metrics and Join Risk

Now suppose the target grain is:

> **one row per department**

We join both admissions and staff to departments.

Each department may have:
- many admissions,
- many staff.

That can multiply rows.

For counts, `COUNT(DISTINCT ...)` can protect the metrics.


## 23. Department summary


In [ ]:
display_query("""
SELECT
    d.department,
    COUNT(DISTINCT a.admission_id) AS admissions,
    COUNT(DISTINCT a.patient_id) AS patients,
    COUNT(DISTINCT s.staff_id) AS staff_count,
    ROUND(
        AVG(a.discharge_date - a.admit_date),
        1
    ) AS avg_stay_days,
    string_agg(
        DISTINCT s.specialization,
        ', '
        ORDER BY s.specialization
    ) AS specializations
FROM departments AS d
LEFT JOIN admissions AS a
    ON a.department = d.department
LEFT JOIN medical_staff AS s
    ON s.department = d.department
GROUP BY d.department
ORDER BY admissions DESC, d.department;
""", title="Department metrics")


### A caution about averages

Even when `COUNT(DISTINCT ...)` protects counts, an average can still be biased if admission rows are repeated once per staff member.

That is why pre-aggregation is often preferable for production analytics.

For learning purposes, this example is useful because it exposes the design question.


# Part XVII · Join Validation

A join is not finished when it runs.

Validate it.


## 24. Check whether an INNER JOIN lost admissions unexpectedly


In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM admissions) AS admission_rows,
    (
        SELECT COUNT(*)
        FROM patients AS p
        JOIN admissions AS a
            ON a.patient_id = p.patient_id
    ) AS joined_rows;
""", title="Source admissions vs joined rows")


If every admission has a valid patient foreign key, these counts should match.


## 25. Find patients with no admission


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.patient_id = p.patient_id
WHERE a.admission_id IS NULL
ORDER BY p.patient_id;
""", title="Unmatched patients")


## 26. Find staging records with no patient master


In [ ]:
display_query("""
SELECT
    s.import_row_id,
    s.patient_id,
    s.diagnosis,
    s.source_file
FROM admissions_stage AS s
LEFT JOIN patients AS p
    ON p.patient_id = s.patient_id
WHERE p.patient_id IS NULL
ORDER BY s.import_row_id;
""", title="Orphan staging rows")


### Join validation checklist

After an important join, ask:

1. What is the expected output grain?
2. How many rows existed before the join?
3. How many rows exist after the join?
4. Were any keys unexpectedly unmatched?
5. Did one-to-many relationships multiply rows?
6. Are counts inflated?
7. Are sums or averages duplicated?
8. Should child tables be pre-aggregated first?


# Part XVIII · Optional Project Pattern: Build a Patient Timeline

Sometimes the correct solution is not a join.

Admissions, labs, and prescriptions are different kinds of events.

If the goal is:

> one row per patient event

then stacking the event tables with `UNION ALL` is more natural than joining them side-by-side.


## 27. Create one patient event timeline


In [ ]:
display_query("""
WITH patient_timeline AS (
    SELECT
        patient_id,
        admit_date AS event_date,
        'Admission'::text AS event_type,
        diagnosis::text AS event_details
    FROM admissions

    UNION ALL

    SELECT
        patient_id,
        test_date,
        'Lab Test',
        test_name || ': ' || COALESCE(result_value::text, 'NULL')
    FROM lab_results

    UNION ALL

    SELECT
        patient_id,
        prescribed_date,
        'Prescription',
        medication || ' - ' || dosage
    FROM prescriptions
)
SELECT
    p.name,
    t.event_date,
    t.event_type,
    t.event_details
FROM patient_timeline AS t
JOIN patients AS p USING (patient_id)
ORDER BY p.name, t.event_date, t.event_type;
""", title="Patient event timeline")


### Why UNION ALL instead of several joins?

The event tables represent different event types.

We want to **stack events vertically**, not combine every lab with every prescription and admission.

This is an important modeling choice:

> Sometimes the right answer is not a join.


# Part XIX · Optional Preview: Department Transitions

This example combines event sequencing and aggregation.

It is included as a preview of techniques that become more important later in the course.


In [ ]:
display_query("""
WITH seq AS (
    SELECT
        patient_id,
        department,
        admit_date,
        LEAD(department) OVER (
            PARTITION BY patient_id
            ORDER BY admit_date
        ) AS next_department,
        LEAD(admit_date) OVER (
            PARTITION BY patient_id
            ORDER BY admit_date
        ) AS next_admit_date
    FROM admissions
)
SELECT
    department AS from_department,
    next_department AS to_department,
    COUNT(*) AS transitions,
    ROUND(
        AVG(next_admit_date - admit_date),
        1
    ) AS avg_days_between
FROM seq
WHERE next_department IS NOT NULL
  AND department IS DISTINCT FROM next_department
GROUP BY department, next_department
ORDER BY transitions DESC, from_department, to_department;
""", title="Department transitions")


# Part XX · Guided Practice

Before writing each query, complete these two sentences:

> **One output row should represent ________.**

> **The population I must preserve is ________.**


## Practice 1 · INNER JOIN

Return one row per admission with:

- patient name
- admission ID
- diagnosis
- department

<details>
<summary>Hint</summary>

Join `patients` to `admissions` on `patient_id`.

</details>


In [ ]:
practice_1 = """
SELECT
    -- columns
FROM patients AS p
JOIN admissions AS a
    ON
    -- match condition
ORDER BY
    -- useful ordering
;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · LEFT JOIN

Return every patient, including patients with no admission.

Show:
- patient ID
- patient name
- admission ID
- diagnosis


In [ ]:
practice_2 = """
SELECT
FROM patients AS p
LEFT JOIN admissions AS a
    ON
ORDER BY p.patient_id;
"""


## Practice 3 · Anti-join

Find patients who have no lab result.

<details>
<summary>Hint</summary>

LEFT JOIN to `lab_results`, then test a non-null child key for NULL.

</details>


In [ ]:
practice_3 = """
SELECT
FROM patients AS p
LEFT JOIN lab_results AS l
    ON
WHERE
    -- no matching child
;
"""


## Practice 4 · ON vs WHERE

Keep every patient, but attach only Glucose lab results.

Put the Glucose condition in the correct place so patients without Glucose tests remain.


In [ ]:
practice_4 = """
SELECT
FROM patients AS p
LEFT JOIN lab_results AS l
    ON
    -- patient match
    -- Glucose condition
ORDER BY p.patient_id;
"""


## Practice 5 · Safe counts

Create one row per patient containing:

- admission count
- lab count
- prescription count

Try two versions:

1. direct joins with `COUNT(DISTINCT ...)`
2. pre-aggregate each child table before joining

Compare the designs.


In [ ]:
practice_5 = """
-- Version 1 or Version 2

"""


## Practice 6 · Department metrics

Create one row per department containing:

- admission count
- distinct patient count
- staff count

Preserve departments with no activity.


In [ ]:
practice_6 = """
SELECT
    d.department,
    -- metrics
FROM departments AS d
-- joins
GROUP BY d.department
ORDER BY d.department;
"""


# Part XXI · Final Challenge

Build a **patient activity feature table** with exactly one row per patient.

Return:

- patient ID
- patient name
- admission count
- lab count
- prescription count
- latest admission date
- latest lab date
- latest prescription date

### Requirement

Do **not** join raw admissions, labs, and prescriptions directly and then aggregate.

Instead:

1. summarize each child table to patient grain,
2. join the summaries to patients,
3. use `COALESCE(..., 0)` for missing counts.

This final challenge tests whether you can protect the target grain.


In [ ]:
final_query = """
WITH admission_summary AS (
    SELECT
        patient_id,
        -- admission count,
        -- latest admission date
    FROM admissions
    GROUP BY patient_id
),
lab_summary AS (
    SELECT
        patient_id,
        -- lab count,
        -- latest lab date
    FROM lab_results
    GROUP BY patient_id
),
rx_summary AS (
    SELECT
        patient_id,
        -- prescription count,
        -- latest prescription date
    FROM prescriptions
    GROUP BY patient_id
)
SELECT
    p.patient_id,
    p.name
    -- joined summary columns
FROM patients AS p
-- join summaries
ORDER BY p.patient_id;
"""

# display_query(final_query, title="Final challenge")


# Quick Concept Check

### What does the ON clause decide?
<details><summary>Show answer</summary>Which rows from the two inputs match each other.</details>

### What does the join type decide?
<details><summary>Show answer</summary>Which unmatched rows should still be preserved.</details>

### Why can a LEFT JOIN return NULL values from the right table?
<details><summary>Show answer</summary>The left row was preserved even though no right-side row matched.</details>

### What is an anti-join pattern?
<details><summary>Show answer</summary>LEFT JOIN to a child table, then keep rows where a non-null child key is NULL.</details>

### Why can a right-table filter in WHERE change a LEFT JOIN result?
<details><summary>Show answer</summary>It can remove the NULL-extended unmatched rows, making the result behave like an inner join for that condition.</details>

### What causes row multiplication?
<details><summary>Show answer</summary>Joining one parent to multiple one-to-many child tables at the same time can create combinations of child rows.</details>

### Why is COUNT(DISTINCT child_id) sometimes needed?
<details><summary>Show answer</summary>The same child row may appear multiple times after another one-to-many table is joined.</details>

### What is often safer than direct multi-child joins for a one-row-per-patient feature table?
<details><summary>Show answer</summary>Pre-aggregate each child table to patient grain, then join the summaries.</details>


# Module 6 Summary

A reliable join workflow is:

1. Define the target grain.
2. Identify the relationship cardinality.
3. Decide which population must be preserved.
4. Write the match condition.
5. Place child filters carefully in `ON` or `WHERE`.
6. Inspect row growth.
7. Watch for multiplied counts and measures.
8. Pre-aggregate child tables when needed.
9. Validate unmatched keys and row counts.
10. Remember that sometimes `UNION ALL` is a better model than a join.

The central question is always:

> **What should one output row represent?**

## Next
**Module 7: Subqueries in PostgreSQL**
